# 应用案例: 模拟土地利用变化的影响

## 目的
水文模型的一个重要应用是评估土地利用变化（如城市化）对流域水文响应的影响。通常，城市化会导致不透水面积增加，从而使得径流产生得更快、更多，洪峰流量也更高。这个过程被称为“洪峰的尖锐化”（flashier hydrograph）。

本教程将作为一个具体的应用案例，展示如何使用本框架来模拟这一过程。

此笔记本将：
1.  建立一个代表“开发前”（例如，乡村或森林）情景的水文模型，其产流参数反映了高渗透性。
2.  建立一个代表“开发后”（例如，城市化）情景的水文模型，其产流参数反映了低渗透性。
3.  在相同的降雨事件下运行这两个情景。
4.  通过对比两个情景的出口流量过程线，来量化和可视化土地利用变化带来的水文效应。

## 1. 环境设置和模型准备

我们使用`SCSCurveNumberModule`作为产流模块，因为它的核心参数`CN`（径流曲线数）与土地利用类型有非常直接的物理联系。一般来说，城市地区的`CN`值要远高于森林或草地。
- **开发前**: 我们使用一个较低的 `CN=65`，代表植被覆盖良好的情况。
- **开发后**: 我们使用一个较高的 `CN=85`，代表有大量不透水面的城市区域。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting

# --- 定义一个辅助函数来运行模型 ---
def run_scenario(cn_value, routing_params, rainfall):
    runoff_module = SCSCurveNumberModule(CN=cn_value)
    routing_module = SimpleRouting(**routing_params)
    model = HydrologicalModel(runoff_module, routing_module)
    
    outflows = [model.run(rain, pet=0) for rain in rainfall]
    return np.array(outflows)

# --- 定义共享的输入和参数 ---
routing_params = {'k_q': 0.6, 'k_s': 0.1}
# 定义一个设计暴雨事件 (例如，24小时暴雨)
rainfall_event = np.array([0, 2, 5, 8, 12, 20, 35, 50, 40, 25, 15, 10, 8, 6, 4, 3, 2, 1, 0, 0, 0, 0, 0, 0])

## 2. 运行两个情景的模拟

我们分别使用`CN=65`和`CN=85`来运行模拟，其他所有条件（汇流参数、降雨）都保持完全一致。

In [ ]:
# 开发前情景
cn_pre = 65
outflow_pre_dev = run_scenario(cn_pre, routing_params, rainfall_event)
print(f"运行'开发前'情景 (CN={cn_pre})...")

# 开发后情景
cn_post = 85
outflow_post_dev = run_scenario(cn_post, routing_params, rainfall_event)
print(f"运行'开发后'情景 (CN={cn_post})...")

print("\n所有情景运行完毕。")

## 3. 结果对比与可视化

我们将两个情景的出口流量过程线，以及作为输入的降雨过程，绘制在同一张图上进行比较。

In [ ]:
timesteps = np.arange(len(rainfall_event))
fig, ax1 = plt.subplots(figsize=(12, 7))

# 绘制流量过程线
color = 'tab:blue'
ax1.set_xlabel('Time (hours)')
ax1.set_ylabel('Discharge (mm/hr)', color=color)
ax1.plot(timesteps, outflow_pre_dev, 'o--', color='green', label=f'Pre-Development (CN={cn_pre})')
ax1.plot(timesteps, outflow_post_dev, 's-', color='darkred', label=f'Post-Development (CN={cn_post})')
ax1.tick_params(axis='y', labelcolor=color)
ax1.legend(loc='upper left')
ax1.grid(True)

# 在第二个Y轴上绘制降雨
ax2 = ax1.twinx()
color = 'tab:gray'
ax2.set_ylabel('Rainfall (mm/hr)', color=color)
ax2.bar(timesteps, rainfall_event, color=color, alpha=0.5, label='Rainfall')
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right')

fig.tight_layout()
plt.title('Impact of Land Use Change (Urbanization) on Runoff Hydrograph')
plt.show()

# --- 量化对比 ---
peak_flow_pre = np.max(outflow_pre_dev)
peak_flow_post = np.max(outflow_post_dev)
total_runoff_pre = np.sum(outflow_pre_dev)
total_runoff_post = np.sum(outflow_post_dev)

print(f"开发前洪峰流量: {peak_flow_pre:.2f} mm/hr")
print(f"开发后洪峰流量: {peak_flow_post:.2f} mm/hr")
print(f"洪峰流量增加了: {((peak_flow_post/peak_flow_pre - 1) * 100):.1f}%")
print("-----")
print(f"开发前总径流量: {total_runoff_pre:.2f} mm")
print(f"开发后总径流量: {total_runoff_post:.2f} mm")
print(f"总径流量增加了: {((total_runoff_post/total_runoff_pre - 1) * 100):.1f}%")

## 4. 结论

从图表和量化结果中可以清晰地看到：
- **洪峰更高**: 开发后的洪峰流量显著高于开发前。这是因为城市化导致的不透水面（如屋顶、道路）使得雨水无法下渗，直接转化为地表径流。
- **响应更快**: 开发后的洪峰出现时间比开发前更早。这同样是因为径流路径更有效率（例如，通过排水管道），缩短了水流汇集到出口的时间。
- **总径流量更大**: 开发后的总径流量也明显增加，这意味着有更少的水下渗补充地下水，更多的水作为洪水流走。

这个简单的案例研究展示了如何使用本框架来评估和量化工程活动或自然变化对水文系统的影响，这是水资源管理和防洪规划中的一个核心任务。